# 15 — Delta Lake Features: Time Travel, History, OPTIMIZE, Z-ORDER (Spark SQL)

Mesmas funcionalidades do notebook `spark_sql/15_delta_features.ipynb`,
demonstradas via Spark SQL em vez da API Python.

| Feature | Para que serve |
|---------|----------------|
| `DESCRIBE HISTORY` | Auditoria: quem fez o quê e quando |
| **Time Travel** | Ler qualquer versão anterior da tabela |
| `OPTIMIZE` | Compactar small files → performance de leitura |
| `Z-ORDER` | Co-localizar dados por coluna → acelerar filtros |


In [1]:
import sys
import os
sys.path.insert(0, os.getcwd())
from utils import get_spark, register_catalog
from delta.tables import DeltaTable

spark = get_spark("NorthwindDW SQL - 15 Delta Features")
register_catalog(spark)
print("Spark:", spark.version)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/29 01:04:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Catálogo registrado: {'bronze': 11, 'silver': 0, 'gold': 15}
Spark: 3.5.0


## 1. DESCRIBE HISTORY — Auditoria de operações


In [2]:
# Histórico completo da DimCustomer via Spark SQL
spark.sql("""
    DESCRIBE HISTORY gold.DimCustomer
""").select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)


+-------+-----------------------+------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation   |operationMetrics                                                                                                                                                                                                                                                                                                              |
+-------+-----------------------+------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 2. Time Travel por versão


In [3]:
# Ler version=0 (carga inicial) via Spark SQL
v0 = spark.sql("SELECT * FROM gold.DimCustomer VERSION AS OF 0")
current = spark.sql("SELECT COUNT(*) AS n FROM gold.DimCustomer").collect()[0]['n']

print(f"Version 0 (carga inicial):  {v0.count()} linhas")
print(f"Versão atual:               {current} linhas")
print(f"Linhas extras (históricas): {current - v0.count()}")
print("\nPrimeiras 5 linhas da version 0:")
spark.sql("""
    SELECT CustomerID, CompanyName, City, IsCurrent, ValidFrom, ValidTo
    FROM gold.DimCustomer VERSION AS OF 0
    LIMIT 5
""").show()


Version 0 (carga inicial):  0 linhas
Versão atual:               92 linhas


Linhas extras (históricas): 92

Primeiras 5 linhas da version 0:


+----------+-----------+----+---------+---------+-------+
|CustomerID|CompanyName|City|IsCurrent|ValidFrom|ValidTo|
+----------+-----------+----+---------+---------+-------+
+----------+-----------+----+---------+---------+-------+



## 3. Time Travel por timestamp


In [4]:
# Informações da última operação
last_op = spark.sql("DESCRIBE HISTORY gold.DimCustomer LIMIT 1").collect()[0]
print(f"Última operação: {last_op['operation']} em {last_op['timestamp']}")
print(f"Versão: {last_op['version']}")
print(f"Métricas: {last_op['operationMetrics']}")


Última operação: WRITE em 2026-03-29 00:48:59.879000
Versão: 3
Métricas: {'numOutputRows': '1', 'numOutputBytes': '3049', 'numFiles': '1'}


## 4. OPTIMIZE — Compactação de small files


In [5]:
# Estado ANTES
before = spark.sql("DESCRIBE DETAIL gold.DimCustomer").select("numFiles", "sizeInBytes").collect()[0]
print(f"ANTES  — numFiles: {before['numFiles']}, size: {before['sizeInBytes']:,} bytes")

# Compactar via SQL
spark.sql("OPTIMIZE gold.DimCustomer")

# Estado DEPOIS
after = spark.sql("DESCRIBE DETAIL gold.DimCustomer").select("numFiles", "sizeInBytes").collect()[0]
print(f"DEPOIS — numFiles: {after['numFiles']}, size: {after['sizeInBytes']:,} bytes")


ANTES  — numFiles: 2, size: 11,608 bytes


DEPOIS — numFiles: 1, size: 8,599 bytes


## 5. Z-ORDER — Co-localização por coluna de filtro frequente


In [6]:
# Z-ORDER em FactSales por OrderDateKey via SQL
spark.sql("OPTIMIZE gold.FactSales ZORDER BY (OrderDateKey)")

spark.sql("DESCRIBE DETAIL gold.FactSales").select("numFiles", "sizeInBytes", "partitionColumns").show(truncate=False)
print("Z-ORDER concluído. Data skipping habilitado para filtros em OrderDateKey.")


+--------+-----------+----------------+
|numFiles|sizeInBytes|partitionColumns|
+--------+-----------+----------------+
|1       |54809      |[]              |
+--------+-----------+----------------+

Z-ORDER concluído. Data skipping habilitado para filtros em OrderDateKey.


## 6. Versões como mecanismo de recuperação


In [7]:
# Todas as versões em ordem
spark.sql("""
    DESCRIBE HISTORY gold.DimCustomer
""").select("version", "timestamp", "operation", "operationMetrics") \
   .orderBy("version").show(truncate=False)

v_current = spark.sql("SELECT COUNT(*) AS n FROM gold.DimCustomer").collect()[0]['n']
v0_count  = spark.sql("SELECT COUNT(*) AS n FROM gold.DimCustomer VERSION AS OF 0").collect()[0]['n']
print(f"\nVersão atual: {v_current} linhas")
print(f"Version 0:    {v0_count} linhas")
print(f"Um RESTORE VERSION AS OF 0 removeria {v_current - v0_count} linhas históricas.")
print("(RESTORE não executado para preservar o estado do portfólio)")


+-------+-----------------------+------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation   |operationMetrics                                                                                                                                                                                                                                                                                                              |
+-------+-----------------------+------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


Versão atual: 92 linhas
Version 0:    0 linhas
Um RESTORE VERSION AS OF 0 removeria 92 linhas históricas.
(RESTORE não executado para preservar o estado do portfólio)
